# Punto 1 — Obtener información del partido

**Proyecto Final · Curso LanusStats** · Lucas Marinelli · @datafutbol_ar

**Partido principal:** Argentina 1–2 Arabia Saudita — debut Mundial 2022 (`match_id 3857300`)

> Consigna: obtener la información de un partido del Mundial 2022 usando la API de
> StatsBomb (la manera directa de la doc **o** el parser de mplsoccer). Acá muestro **las dos**.

## Por qué este partido

Elegí el **debut** y no la final a propósito. La final 2022 está hipersaturada de
análisis: entregar eso es "uno más". El debut, en cambio, lo analizó casi nadie y tiene
una historia mejor para contar: **el campeón perdió su primer partido**. Argentina
generó más juego pero cayó 1–2 — la paradoja perfecta para arrancar un análisis.

Además me sirve para el bonus: comparar este partido con la final (mismo equipo, mismo
jugador estrella, resultado opuesto).

## Setup

`from helpers import *` trae rutas, estilo de marca (Combo C) y las funciones de carga
con cache. Si ya corriste `00_setup.ipynb`, los eventos se leen del parquet (instantáneo).

In [1]:
%load_ext autoreload
%autoreload 2
from helpers import *

## Método A — Manera directa (`statsbombpy`)

Es la forma de la documentación: `sb.events(match_id=...)`. Nuestro helper `cargar_eventos`
la envuelve con cache (baja una vez, después lee del disco).

In [2]:
ev = cargar_eventos(MATCH_ARG_SAU, 'arg_sau')
print('shape:', ev.shape)   # (filas=eventos, columnas=variables)
ev[['minute','second','team','player','type']].head(10)

[cache] arg_sau: 3329 eventos leidos de eventos_arg_sau.parquet
shape: (3329, 93)


,minute,second,team,player,type
0,0,0,Argentina,NaN,Starting XI
1,0,0,Saudi Arabia,NaN,Starting XI
2,0,0,Saudi Arabia,NaN,Half Start
3,0,0,Argentina,NaN,Half Start
4,45,0,Saudi Arabia,NaN,Half Start
5,45,0,Argentina,NaN,Half Start
6,0,0,Saudi Arabia,Saleh Khalid Al Shehri,Pass
7,0,2,Saudi Arabia,Mohammed Kanoo,Pass
8,0,3,Saudi Arabia,Saleh Khalid Al Shehri,Pass
9,0,6,Saudi Arabia,Mohammed Kanoo,Pass


### Datos del partido (marcador, fecha, estadio)

In [3]:
partidos = lista_partidos()
fila = partidos[partidos['match_id'] == MATCH_ARG_SAU].iloc[0]

print(f"{fila['home_team']} {fila['home_score']} - {fila['away_score']} {fila['away_team']}")
print('Fecha   :', fila['match_date'])
print('Estadio :', fila.get('stadium', 's/d'))
print('Fase    :', fila.get('competition_stage', 's/d'))

Argentina 1 - 2 Saudi Arabia
Fecha   : 2022-11-22
Estadio : Lusail Stadium
Fase    : Group Stage


## Método B — Parser de mplsoccer (`Sbopen`)

mplsoccer trae su propio lector de los datos abiertos de StatsBomb. Devuelve 4 tablas:
`event` (eventos), `related` (eventos relacionados), `freeze` (posiciones en los tiros) y
`tactics` (alineaciones). Es la otra forma que menciona la consigna.

In [4]:
from mplsoccer import Sbopen

parser = Sbopen()
df_ev, df_related, df_freeze, df_tactics = parser.event(MATCH_ARG_SAU)
print('eventos (parser):', df_ev.shape)
df_ev[['minute','team_name','player_name','type_name']].head(10)

eventos (parser): (3329, 76)


,minute,team_name,player_name,type_name
0,0,Argentina,NaN,Starting XI
1,0,Saudi Arabia,NaN,Starting XI
2,0,Saudi Arabia,NaN,Half Start
3,0,Argentina,NaN,Half Start
4,0,Saudi Arabia,Saleh Khalid Al Shehri,Pass
5,0,Saudi Arabia,Mohammed Kanoo,Ball Receipt
6,0,Saudi Arabia,Mohammed Kanoo,Carry
7,0,Saudi Arabia,Mohammed Kanoo,Pass
8,0,Saudi Arabia,Saleh Khalid Al Shehri,Ball Receipt
9,0,Saudi Arabia,Saleh Khalid Al Shehri,Carry


### Diferencia clave entre los dos métodos

Traen **los mismos eventos**, pero nombran distinto las columnas:

| Dato | `statsbombpy` (Método A) | `Sbopen` (Método B) |
|---|---|---|
| Equipo | `team` | `team_name` |
| Jugador | `player` | `player_name` |
| Tipo de acción | `type` | `type_name` |
| Coordenadas | `location` (lista `[x,y]`) | columnas `x`, `y` separadas |

**Para el resto del proyecto uso el Método A** (`statsbombpy` + cache), porque ya lo
tenemos cacheado y es el que envuelven nuestros helpers.

## Inspección — ¿qué tiene esta tabla?

Antes de los siguientes puntos, conviene conocer el contenido (ver `manual_busqueda_basica.md §3.6`).

In [6]:
# Tipos de evento más frecuentes del partido
ev['type'].value_counts().head(20)

type
Pass               932
Ball Receipt*      874
Carry              755
Pressure           248
Ball Recovery       93
Duel                61
Clearance           47
Block               32
Interception        32
Foul Committed      29
Miscontrol          28
Foul Won            28
Dribble             27
Goal Keeper         23
Dispossessed        19
Shot                18
50/50               18
Dribbled Past       14
Injury Stoppage     11
Substitution         9
Name: count, dtype: int64

## Conclusión — Punto 1 ✅

Cargué el debut de Argentina en el Mundial 2022 de **dos formas** (statsbombpy directo y
parser Sbopen de mplsoccer) y verifiqué el partido: **Argentina 1–2 Arabia Saudita**, en
el Lusail Stadium. Los eventos quedan cacheados en `data/eventos_arg_sau.parquet` para
reusar en los puntos 2 a 5.

➡️ Sigue: **`punto_2.ipynb`** — estadísticas del partido (xG por equipo, líder de pases,
líder de recuperaciones, tercio con más toques).